<a href="https://colab.research.google.com/github/fandris03/ResNet-project/blob/main/ResNet_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ResNet - CIFAR-10

##Setup

In [39]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from google.colab import drive

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


In [31]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [32]:
!ls /content/drive/MyDrive

 Benl		    datasets				        my_models
 BeNL		    DRL_03_Imitation_Learning_completed.ipynb
'Colab Notebooks'   KÉPEK


In [33]:
!ls -la /content/drive/MyDrive/datasets
!ls /content/drive/MyDrive/datasets/cifar-10-batches-py

total 166515
drwx------ 3 root root      4096 Sep 29 08:26 .
drwx------ 9 root root      4096 Sep 29 08:19 ..
drwx------ 2 root root      4096 Sep 29 09:48 cifar-10-batches-py
-rw------- 1 root root 170498071 Sep 29 09:33 cifar-10-python.tar.gz
 batches.meta   data_batch_2   data_batch_4  'data_batch_5 (1)'   test_batch
 data_batch_1   data_batch_3   data_batch_5   readme.html


In [34]:
#drive.mount('/content/drive')
data_root = "/content/drive/MyDrive/datasets"

transforms_cifar = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = torchvision.datasets.CIFAR10(
    root=data_root,
    train=True,
    download=True,
    transform=transforms_cifar
)

test_dataset = torchvision.datasets.CIFAR10(
    root=data_root,
    train=False,
    download=True,
    transform=transforms_cifar
)

In [35]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=2)

#Model

In [44]:
class ShortcutA(nn.Module):
    '''
    Parameter-free identity shortcut
    Subsamples spatially with stride, then zero-pads the channel dimension.
    '''
    def __init__(self, in_channels, out_channels, stride):
        super().__init__()
        self.stride = stride
        self.pad_channels = out_channels - in_channels

    def forward(self, x):
        if self.stride != 1:
            x = x[:, :, ::self.stride, ::self.stride]  # spatial subsample #TRY MaxPool/AvgPool
        if self.pad_channels > 0:
            # (W_left, W_right, H_top, H_bottom, C_front, C_back)
            x = F.pad(x, (0, 0, 0, 0, 0, self.pad_channels))
        return x


class ResidualBlock(nn.Module):
    '''
    Residual block of 2 conv layers with BatchNorm and ReLU for each layer
    and a skip connection

    bias negligated by batchnorm => False
    '''
    def __init__(self, in_channels, out_channels, stride=1, projection=False):  #projection depends on stride, refactor!!!
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3,
                                stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                                padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        # Projection shortcut, only needed if shape changes
        self.shortcut = nn.Sequential()
        if projection:
            self.shortcut = nn.Sequential(
                    nn.Conv2d(in_channels, out_channels,
                              kernel_size=1,stride=stride,
                              bias=False),
                    nn.BatchNorm2d(out_channels)
                )

    def forward(self, x):
        identity = self.shortcut(x)

        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        out += identity
        out = self.relu(out)

        return out



class DeepNN(nn.Module):
    '''
    ResNet-32
    5 blocks, 32 layers total
    '''
    def __init__(self, num_classes=10):
        super(DeepNN, self).__init__()
        self.features = nn.Sequential(
            #Stage 1
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            #Stage 2 (conv2_x)
            ResidualBlock(16, 16),
            ResidualBlock(16, 16),
            ResidualBlock(16, 16),
            ResidualBlock(16, 16),
            ResidualBlock(16, 16),
            #Stage 3
            ResidualBlock(16, 32, stride=2, projection=True),
            ResidualBlock(32, 32),
            ResidualBlock(32, 32),
            ResidualBlock(32, 32),
            ResidualBlock(32, 32),
            #Stage 4
            ResidualBlock(32, 64, stride=2, projection=True),
            ResidualBlock(64, 64),
            ResidualBlock(64, 64),
            ResidualBlock(64, 64),
            ResidualBlock(64, 64),
            #stage 5
            nn.AvgPool2d(8, stride=1),
            #nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(64, 10)
            )
        '''
        self.classifier = nn.Sequential(
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, num_classes)
            )
        '''

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        #x = self.classifier(x)
        return x